# Merge spikes that were automatically detected from different templates

Load packages

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import scipy
import glob
from scipy import signal
import os
from ephyviewer import mkQApp, MainViewer, TraceViewer
from scipy.stats import zscore
from ephyviewer import mkQApp, MainViewer, TraceViewer, CsvEpochSource, InMemoryAnalogSignalSource, EpochEncoder,TimeFreqViewer,AnalogSignalSourceWithScatter, EpochEncoder_ABC
from matplotlib import cm
from matplotlib.colors import to_hex
import re
import mne
import pandas as pd
import numpy as np
import csv
from collections import defaultdict
import ast
import matplotlib
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import string
import pickle
from IPython.display import display
from ipyfilechooser import FileChooser
import ipywidgets as widgets
%matplotlib widget

Choose .edf file

In [ ]:
dpath = "//10.69.168.1/crnldata/forgetting/Aurelie/EpiKid/CleanedData/"
fc1 = FileChooser(dpath,select_default=True, show_only_dirs = False, title = "<b>Choose a .edf file </b>", layout=widgets.Layout(width='100%'))
fc1.filter_pattern = '*repaired_montage.edf'
display(fc1)
def update_my_folder(chooser):
    global dpath
    dpath = chooser.selected
    %store dpath
    return 
fc1.register_callback(update_my_folder)

Load channel infos

In [ ]:
eeg_path= Path(dpath)
eeg_file=('_').join(eeg_path.name.split("_")[:2])
spikes_file_1 = f"//10.69.168.1/crnldata/forgetting/Aurelie/EpiKid/SpikeDet/20260220_150302_positivespikes/sp_spikes_spmeeg_{eeg_file}_repaired_montage.mat"  #glob.glob(f'{folder_base}/sp_spikes*_repaired_montage.mat')
spikes_file_2 = f"//10.69.168.1/crnldata/forgetting/Aurelie/EpiKid/SpikeDet/20260223_095647_negativespikes/sp_spikesNeg_spmeeg_{eeg_file}_repaired_montage.mat"  #glob.glob(f'{folder_base2}/sp_spikes*_repaired_montage.mat')

data = mne.io.read_raw_edf(eeg_path, preload=False)
channels = data.ch_names
samplerate = data.info.get('sfreq')  # in Hz
montage_name=[]
for i, ch in enumerate(channels):
    if ch.count('-')==0:
        montage_name.append(ch)
    else:        
        montage_name.append(ch)
montage_name_eeg=montage_name[:-2]

Merge detected spikes & cluster identities

In [ ]:
matsp_1 = scipy.io.loadmat(spikes_file_1, squeeze_me=True, struct_as_record=False)
sp_array_1 = matsp_1['sp']
matsp_2 = scipy.io.loadmat(spikes_file_2, squeeze_me=True, struct_as_record=False)
sp_array_2 = matsp_2['sp']

sp_array_both = dict()
cluster_both = dict()
for ch, channel in enumerate(montage_name_eeg):
    spikes_1 = np.array(sp_array_1[ch].GenDet.Epoch)
    spikes_2 = np.array(sp_array_2[ch].GenDet.Epoch)
    cluster_1 = np.array(sp_array_1[ch].Clusters.ClusterIndices)
    cluster_2 = np.array(sp_array_2[ch].Clusters.ClusterIndices) + max(cluster_1)  # Shift cluster indices of the second array
    merged_spikes = np.concatenate([spikes_1, spikes_2])
    sort_indices = np.argsort(merged_spikes)
    spikes_sorted_merged = merged_spikes[sort_indices]
    clusters_merged = np.concatenate([cluster_1, cluster_2])
    clusters_sorted = clusters_merged[sort_indices]

    # Remove values that are too close (< 200 ms apart)
    min_distance = 0.2 * samplerate 
    keep_mask = np.ones(len(spikes_sorted_merged), dtype=bool)
    for i in range(1, len(spikes_sorted_merged)):
        if spikes_sorted_merged[i] - spikes_sorted_merged[i-1] < min_distance:
            keep_mask[i] = False  # Remove the second one
    spikes_sorted_merged_filtered = spikes_sorted_merged[keep_mask]
    clusters_sorted_filtered = clusters_sorted[keep_mask]

    sp_array_both[ch] = spikes_sorted_merged_filtered
    cluster_both[ch] = clusters_sorted_filtered

Create scatter channels & indexes for ephyviewer

In [ ]:
scatter_channels_both = defaultdict(list)
scatter_indexes_both  = defaultdict(list)
nb_cluster_tot_both = 0
for i, channel in enumerate(montage_name_eeg):
    spikes = np.array(sp_array_both[i])
    cluster = np.array(cluster_both[i])
    nb_cluster = int(cluster.max())
    for clus in np.arange(nb_cluster_tot_both, nb_cluster_tot_both + nb_cluster, 1):
        scatter_channels_both[clus+1]=[np.repeat(np.arange(0,len(montage_name_eeg)), nb_cluster)[clus]]
        scatter_indexes_both[clus+1]=[]
    nb_cluster_tot_both += nb_cluster

for i, channel in enumerate(montage_name_eeg):
    spikes = np.array(sp_array_both[i])
    cluster = np.array(cluster_both[i])
    for spike in np.arange(0, len(cluster), 1):    
        key = int(cluster[spike] + ((i+1) * nb_cluster)- nb_cluster)
        scatter_indexes_both[key].append(spikes[spike]) 
scatter_channels_both = dict(sorted(scatter_channels_both.items()))
scatter_indexes_both = {k: pd.Series(v) for k, v in scatter_indexes_both.items()}
scatter_indexes_both = dict(sorted(scatter_indexes_both.items()))

Display merged data

In [ ]:
groups = {}
for k, v in scatter_channels_both.items():
    groups.setdefault(v[0], []).append(k)
color_dict = {}
cmap = matplotlib.colormaps['rainbow']
for group_keys in groups.values():
    n = len(group_keys)
    norm = mcolors.Normalize(vmin=0, vmax=n-1)
    for i, key in enumerate(group_keys):
        color_dict[key] = mcolors.to_hex(cmap(norm(i))) + "80"

scatter_indexesO = scatter_indexes_both.copy()
scatter_channelsO = scatter_channels_both.copy()
color_dictO = color_dict.copy()

SleepScoring_filename = eeg_path.parent / f"EphyViewer_Scoring_{eeg_file}.csv"
SleepScoring = pd.read_csv(SleepScoring_filename)

Montage eeg

In [ ]:
raw_data = data.get_data() 
combined = raw_data.T
montage_eeg=[]
montage_name=[]
for i, ch in enumerate(channels):
    if ch.count('-')==0:
        ch1 = ch
        eeg=combined[:,i]
        #montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        montage_eeg = (eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, (eeg[:,np.newaxis]), axis=1)
        montage_name.append(ch)
    else:        
        ch1 = ch
        eeg=combined[:,i]
        nyq = samplerate / 2
        f_lowcut = 0.5
        f_hicut = 50.0 #70
        Wn = [f_lowcut/nyq, f_hicut/nyq]
        sos = signal.butter(6, Wn, btype='band', output='sos')
        eeg = signal.sosfiltfilt(sos, eeg)
        #montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        montage_eeg = (eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, (eeg[:,np.newaxis]), axis=1)
        montage_name.append(ch)
montage_name_eeg=montage_name[:-2]

Remove spikes detected during Wake or Unstaged (optional)

In [ ]:
def find_stages_vectorized(time_values, df):
    df = df.copy()
    df['end_time'] = df['time'] + df['duration']
    # Convert to numpy arrays for faster operations
    times = time_values.values if hasattr(time_values, 'values') else np.array(time_values)
    df_times = df['time'].values
    df_end_times = df['end_time'].values
    df_labels = df['label'].values
    
    # Vectorized comparison using broadcasting
    # Shape: (n_times, n_intervals)
    in_interval = (times[:, np.newaxis] >= df_times) & (times[:, np.newaxis] < df_end_times)
    # Find first matching interval for each time
    matches = np.argmax(in_interval, axis=1)
    has_match = in_interval.any(axis=1)
    # Get labels (None where no match)
    stages = np.where(has_match, df_labels[matches], None)
    return stages

# Optimized main code
labels_to_remove = {None, 'WAKE'}
# Process all keys at once
scatter_indexes = {}
for key, time_list in scatter_indexesO.items():
    # Convert to time in seconds
    times_sec = np.array(time_list) / samplerate
    # Get stages for all times at once
    stages = find_stages_vectorized(times_sec, SleepScoring)
    # Filter out unwanted labels
    mask = ~np.isin(stages, list(labels_to_remove))
    filtered_times = np.array(time_list)[mask]
    scatter_indexes[key] = pd.Series(filtered_times)

# Sort dictionary by keys
scatter_indexes = dict(sorted(scatter_indexes.items()))
scatter_indexesO = scatter_indexes.copy()

Select a cluster (optional)

In [ ]:
sel_cluster = 10  # 1 = purple; 2 = dark blue; 3 = light blue; 4 = cyan; 5 = light green; 6 = light blue; 7 = light yellow; 8 = light orange; 9 = light orange-red; 10 = red
sel_clusters= np.arange(sel_cluster, nb_cluster_tot_both, nb_cluster)
scatter_indexes_both = {k: v for k, v in scatter_indexesO.items() if k in sel_clusters}
scatter_channels_both= {k: v for k, v in scatter_channelsO.items() if k in sel_clusters}
color_dict= {k: v for k, v in color_dictO.items() if k in sel_clusters}
print(f"Cluster {sel_cluster} contains {sum(len(v) for v in scatter_indexes_both.values())} detected spikes")

Ephyviewer

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source = AnalogSignalSourceWithScatter(montage_eeg, samplerate, t_start, scatter_indexes_both, scatter_channels_both,  scatter_colors = color_dict, channel_names=montage_name)
view1 = TraceViewer(source=source)
#scatter_colors = color_dict,
view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#0000ff"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'
